In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy  as np
import ClassicalModel as CM
import torch
import models_parallel as models
import random


def fun(x):
    return np.heaviside(x,0)
def fun2(x):
    return np.sin(np.pi*x)

input_eval  = np.linspace(-1,1,200)
y_true      = fun(input_eval)
y_true2     = fun2(input_eval)

## for layer

The best layer,seed, results from classical 

In [2]:
activation_fuction = [torch.nn.Tanh(),torch.nn.Sigmoid(),torch.nn.ReLU()]
name_function = ["Tanh","Sigmoid","Relu"]

# Classico
for i in range(len(activation_fuction)):
    menor_pred = y_true2.reshape(-1,1)
    erro        = 1
    number_of_layers = 0 
    sample_index = 0
    for N_of_layer in range(1,6):
        for manualSeed in range(1,100):
                   
            np.random.seed(manualSeed)
            random.seed(manualSeed)
            torch.manual_seed(manualSeed)
                
            modelo  = CM.ClassicalModel(
                epochs      = 1000,
                neuronio    = [1]*N_of_layer,
                activation  = activation_fuction[i],
                lr          = 0.01
            )
            modelo.load_checkpoint(checkpoint_path=f"Data/classical/sino_comparing_by_layer/seed{manualSeed}_layer{N_of_layer}_{name_function[i]}")
            predict = modelo.evaluate(input_eval.reshape(-1,1)).detach().numpy()
            erro_t  = np.mean(((predict-y_true2.reshape(-1,1))**2))
            if erro_t < erro:
                erro = erro_t
                menor_pred = predict
                number_of_layers = N_of_layer
                sample_index = manualSeed
    print(f"{name_function[i]} - Erro: ",erro," Nº of layer:",number_of_layers ," sample_index:",sample_index)
 
print("\n\n")
print("Now for Heaviside function:\n") 
# Classico
for i in range(len(activation_fuction)):
    menor_pred = y_true2.reshape(-1,1)
    erro        = 1
    number_of_layers = 0 
    sample_index = 0
    for N_of_layer in range(1,6):
        for manualSeed in range(1,100):
                   
            np.random.seed(manualSeed)
            random.seed(manualSeed)
            torch.manual_seed(manualSeed)
                
            modelo  = CM.ClassicalModel(
                epochs      = 1000,
                neuronio    = [1]*N_of_layer,
                activation  = activation_fuction[i],
                lr          = 0.01
            )
            modelo.load_checkpoint(checkpoint_path=f"Data/classical/Heaviside_comparing_by_layer/seed{manualSeed}_layer{N_of_layer}_{name_function[i]}")
            predict = modelo.evaluate(input_eval.reshape(-1,1)).detach().numpy()
            erro_t  = np.mean((predict-y_true.reshape(-1,1))**2)
            if erro_t < erro:
                erro = erro_t
                menor_pred = predict
                number_of_layers = N_of_layer
                sample_index = manualSeed
    print(f"{name_function[i]} - Erro: ",erro," Nº of layer:",number_of_layers ," sample_index:",sample_index)


Tanh - Erro:  0.06616946224408521  Nº of layer: 5  sample_index: 77
Sigmoid - Erro:  0.06626411324988467  Nº of layer: 5  sample_index: 48
Relu - Erro:  0.06480299534041704  Nº of layer: 3  sample_index: 56



Now for Heaviside function:

Tanh - Erro:  0.0003102706465896743  Nº of layer: 5  sample_index: 34
Sigmoid - Erro:  0.00019000881230560651  Nº of layer: 5  sample_index: 50
Relu - Erro:  0.003967027619826377  Nº of layer: 2  sample_index: 38


## For parameter

In [3]:
from itertools import product
activation_fuction = [torch.nn.Tanh(),torch.nn.Sigmoid(),torch.nn.ReLU()]
name_function = ["Tanh","Sigmoid","Relu"]
def all_neuro(N_layer,numberofneuro):
    todas_combinacoes = []
    for n in range(1, N_layer + 1):
        todas_combinacoes.extend(product(range(1, numberofneuro + 1), repeat=n))

    todas_combinacoes_util=[]
    N_of_parameter = []
    index_all_combinatino = []
    for indice in range(len(todas_combinacoes)):
        modelo  = CM.ClassicalModel(
                epochs      = 1000,
                neuronio    = todas_combinacoes[indice],
                activation  = activation_fuction[0],
                lr          = 0.01) 
        if modelo.Number_of_parameter() in (5,10,15,20,25):
            todas_combinacoes_util.append( todas_combinacoes[indice])
            N_of_parameter.append(modelo.Number_of_parameter())
            index_all_combinatino.append(indice)
            
    # dt_Parametre = [ 
    #                 [3],
    #                 [1, 4],
    #                 [1, 2, 2],
    #                 [1, 1, 1, 2, 1],
    #                 [4, 1, 2],
    #                 [3, 1, 3],
    #                 [1, 1, 5],
    #                 [2, 2, 1, 2],
    #                 [8],
    #                 [5,8],
    #                 [1, 2, 1, 5],
    #                 [1, 2, 2, 1, 3],  
    #                 [1, 1, 2, 2, 2, 1],    
    #                 [4, 1, 2, 3], 
    #                 [2, 2, 3, 2],
    #                 [1, 1, 3, 1, 5],
    #                 [1, 1, 2, 2, 2, 1, 2],
    #                 [3, 2, 5],
    #                 [2, 5, 1, 3],
    #                 [5, 1, 3, 2, 1],
    #                 [2, 1, 2, 2, 1, 1, 1, 2, 1, 1]]
    # for indice in range(len(dt_Parametre)):
    #     modelo  = CM.ClassicalModel(
    #             step_size   = 500,
    #             epochs      = 1000,
    #             neuronio    = todas_combinacoes[indice],
    #             activation  = activation_fuction[0],
    #             lr          = 0.01) 
    #     if modelo.Number_of_parameter() in (5,10,15,20,25):
    #         todas_combinacoes_util.append( dt_Parametre[indice])
    return todas_combinacoes_util,N_of_parameter,index_all_combinatino
combination,number_combination_parameter,index_combination = all_neuro(3,10)      

The best layer,seed, results from classical 

In [4]:
activation_fuction = [torch.nn.Tanh(),torch.nn.Sigmoid(),torch.nn.ReLU()]
name_function = ["Tanh","Sigmoid","Relu"]

# Classico
for index in range(len(name_function)):
    menor_pred  = y_true2.reshape(-1,1)
    erro        = 1
    number_of_i = 0 
    sample_index = 0
    for indice in range(len(combination)):
        ram = []
        for manualSeed in range(1,100):  
                                   
            modelo  = CM.ClassicalModel(
                epochs      = 10000,
                neuronio    = combination[indice],
                activation  = activation_fuction[index],
                lr          = 0.01)
            modelo.load_checkpoint(checkpoint_path=f"Data/classical/sino_comparing_by_parameter/seed{manualSeed}_indice{indice}_{name_function[index]}")
            predict = modelo.evaluate(input_eval.reshape(-1,1)).detach().numpy()
            erro_t  = np.mean(((predict-y_true2.reshape(-1,1))**2))
            if erro_t < erro:
                erro = erro_t
                menor_pred = predict
                number_of_i = indice
                sample_index = manualSeed
    print(f"{name_function[index]} - Erro: ",erro,"indice " , number_of_i ," sample_index:",sample_index)
 
print("\n\n")
print("Now for Heaviside function:\n")
#Classico
for index in range(len(name_function)):
    menor_pred     = y_true2.reshape(-1,1)
    erro           = 1
    number_of_i    = 0 
    sample_index   = 0
    for indice in range(len(combination)):
        ram = []
        for manualSeed in range(1,100):  
                                   
            modelo  = CM.ClassicalModel(
                epochs      = 10000,
                neuronio    = combination[indice],
                activation  = activation_fuction[index],
                lr          = 0.01)
            modelo.load_checkpoint(checkpoint_path=f"Data/classical/Heaviside_comparing_by_parameter/seed{manualSeed}_indice{indice}_{name_function[index]}")
            predict = modelo.evaluate(input_eval.reshape(-1,1)).detach().numpy()
            erro_t  = np.mean((predict-y_true.reshape(-1,1))**2)
            if erro_t < erro:
                erro            = erro_t
                menor_pred      = predict
                number_of_i     = indice
                sample_index    = manualSeed
    print(f"{name_function[index]} - Erro: ",erro,"indice " , number_of_i ," sample_index:",sample_index)


Tanh - Erro:  1.870451445671926e-08 indice  1  sample_index: 88
Sigmoid - Erro:  8.607870104111379e-08 indice  1  sample_index: 23
Relu - Erro:  0.00035180078644863363 indice  1  sample_index: 1



Now for Heaviside function:

Tanh - Erro:  0.0008519623416184352 indice  13  sample_index: 48
Sigmoid - Erro:  0.0011093730699352378 indice  10  sample_index: 39
Relu - Erro:  0.0031215887326401805 indice  3  sample_index: 95


In [5]:

# color =  ['b','g','m']
# # Classical
# index =0
# shift =-0.5
# for index in range(len(name_function)):
#     erro =[] 
#     for indice in range(len(combination)):
#         ram = []
#         for manualSeed in range(1,100):      
#             modelo  = CM.ClassicalModel(
#                 #step_size   = 500,
#                 epochs      = 10000,
#                 neuronio    = combination[indice],
#                 activation  = activation_fuction[index],
#                 lr          = 0.01)
#             modelo.load_checkpoint(checkpoint_path=f"Data/classical/sino_comparing_by_parameter/seed{manualSeed}_indice{indice}_{name_function[index]}")
#             predict = modelo.evaluate(input_eval.reshape(-1,1)).detach().numpy()
#             ram.append( np.mean((predict-y_true2.reshape(-1,1))**2))
#         erro.append(ram)

#     erro    = np.array(erro)  
#     media   = erro.mean()
#     print(media)


# # Quântico
# erro = []
# for j in range(5,6):
#     ram  = []
#     for i in range(1,100):
#         quantum = models.Train(D= 30,number_of_layers=j)
#         quantum.load_checkpoint(checkpoint_path=f'Data/quantum/{j}layer/model_sin{i}.pth')
#         predict = quantum.evaluate(input= input_eval)
#         ram.append(np.mean((predict-y_true2)**2))
#     erro.append(ram) 
    
# erro    = np.array(erro)  
# media   = erro.mean()

# print(media)
